# TMDB Dataset Cleaning & Exploratory Data Analysis

## Objective
This notebook performs data cleaning, normalization, and exploratory analysis on the TMDB Movie Dataset.


---

## 1. Data Loading

This section loads the TMDB movie dataset or generates synthetic data as a fallback.

In [1]:
import pandas as pd
import numpy as np
import os

DATASET_PATH = '../TMDB_movie_dataset_v11.csv'
SAMPLE_SIZE  = 50000

if os.path.exists(DATASET_PATH):
    print(f'Loading dataset from: {DATASET_PATH}')
    df_raw = pd.read_csv(DATASET_PATH, low_memory=False)
    df = df_raw.sample(n=min(SAMPLE_SIZE, len(df_raw)), random_state=42).reset_index(drop=True)
    print(f'Loaded {len(df_raw):,} rows. Working sample: {len(df):,} rows.')
else:
    print('Dataset not found — generating synthetic dataset.')
    np.random.seed(42)
    n = 5000
    GENRES = ['Action','Comedy','Drama','Horror','Thriller','Romance','Sci-Fi','Animation','Documentary','Fantasy']
    LANGS  = ['en','fr','es','de','ja','hi','ko','pt','zh','it']
    OVERVIEWS = [
        'A hero embarks on a dangerous journey to save the world.',
        'Two strangers meet and fall in love against all odds.',
        'A detective unravels a conspiracy in a futuristic city.',
        'An animated tale of friendship, courage and sacrifice.',
        'Aliens invade Earth and only one team can stop them.',
        'A young wizard discovers their true powers at school.',
        'A family torn apart by war seeks reunion across borders.',
        'Corporate espionage leads to a thrilling chase thriller.',
        'Space explorers find a mysterious signal from deep space.',
        'A comedy of errors involving mistaken identity.',
    ]
    df = pd.DataFrame({
        'id':          range(1, n+1),
        'title':       [f'Movie {i:05d}' for i in range(1, n+1)],
        'genres':      [', '.join(np.random.choice(GENRES, np.random.randint(1,4), replace=False)) for _ in range(n)],
        'overview':    [np.random.choice(OVERVIEWS) for _ in range(n)],
        'popularity':  np.abs(np.random.normal(20, 40, n)).round(2),
        'vote_average':np.random.uniform(2.0, 9.8, n).round(1),
        'vote_count':  np.random.randint(0, 15000, n),
        'release_date':pd.date_range('1960-01-01', periods=n, freq='4D').astype(str),
        'revenue':     (np.abs(np.random.exponential(3e7, n))).astype(int),
        'runtime':     np.random.randint(60, 220, n),
        'original_language': np.random.choice(LANGS, n),
        'keywords':    ['adventure hero journey magic' for _ in range(n)],
        'status':      np.random.choice(['Released','Post Production','In Production','Planned'], n,
                                         p=[0.85,0.08,0.05,0.02]),
    })
    df.loc[np.random.choice(df.index, 400, replace=False), 'overview'] = np.nan
    df.loc[np.random.choice(df.index, 200, replace=False), 'runtime']  = np.nan
    df.loc[np.random.choice(df.index, 300, replace=False), 'revenue']  = 0
    print(f'Synthetic dataset shape: {df.shape}')

df.head()

Loading dataset from: ../TMDB_movie_dataset_v11.csv


Loaded 1,400,340 rows. Working sample: 50,000 rows.


,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,original_title,overview,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages,keywords
0,54778,Sea of Death,4.2,13,Released,2009-09-02,0,98,False,NaN,...,Tod aus der Tiefe,In the German North Sea a new cellular life fo...,1.678,/122qMiSuKlKlwzjUnb4iABxbhGX.jpg,NaN,"Science Fiction, Thriller, Drama, TV Movie, Ac...","Sat.1, Crazy Film, Epo-Film, ORF","Germany, Austria",German,"disaster, virus, pandemic"
1,783222,Fuengirola,0.0,0,Released,1967-01-01,0,13,False,NaN,...,Fuengirola,Short documentary by the basque filmmaker Chum...,0.600,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,51411,Ma and Pa Kettle Go to Town,6.1,10,Released,1950-04-01,0,79,False,/ko2HeQbRK1OqHTAOji0ufpHvXVp.jpg,...,Ma and Pa Kettle Go to Town,"When Pa wins a jingle-writing contest, he and ...",2.557,/sbtKSsHZsCVmVBTeI0XV1RtP7RF.jpg,The NEWEST and hilarious adventure!,Comedy,Universal International Pictures,United States of America,English,NaN
3,895452,Anal Centerfolds,9.0,1,Released,2017-01-01,0,171,True,NaN,...,Anal Centerfolds,Anal Centerfolds collects flashy models and pi...,1.169,/2y0JYneySeqoeih1WHX23D6VGtF.jpg,NaN,NaN,Evil Angel,NaN,NaN,NaN
4,1239592,Amateurs Caught On Cam 19,0.0,0,Released,NaN,0,122,True,/vX6YQ8czLA0MdBCy1aqT2aRHXbb.jpg,...,Amateurs Caught On Cam 19,Net Video Girls brings you another helping of ...,0.000,/A2oP1IIu5ai5cndolXnZLfHALb8.jpg,NaN,NaN,NaN,NaN,English,NaN


---

## 2. Data Cleaning

This section cleans duplicate titles, fills missing data, parses dates, and handles outliers.

In [2]:
movies = df.copy()

# Data Cleaning
before = len(movies)
movies.drop_duplicates(subset=['title'], keep='first', inplace=True)
movies.reset_index(drop=True, inplace=True)
print(f"Duplicates removed   : {before - len(movies):,}")

keep_cols = [c for c in ['id','title','genres','overview','popularity','vote_average',
                         'vote_count','release_date','revenue','runtime','original_language','status'] if c in movies.columns]
movies = movies[keep_cols]

movies['overview']   = movies['overview'].fillna('No overview available')
movies['genres']     = movies['genres'].fillna('Unknown')
movies['popularity'] = movies['popularity'].fillna(movies['popularity'].median())
movies['vote_average'] = movies['vote_average'].fillna(movies['vote_average'].median())
movies['vote_count'] = movies['vote_count'].fillna(0)
if 'runtime' in movies.columns:
    movies['runtime'] = movies['runtime'].fillna(movies['runtime'].median())
if 'revenue' in movies.columns:
    movies['revenue'] = movies['revenue'].fillna(0)
movies.dropna(subset=['title'], inplace=True)

movies['release_date'] = pd.to_datetime(movies['release_date'], errors='coerce')
movies['year']  = movies['release_date'].dt.year
movies['month'] = movies['release_date'].dt.month
for col in ['popularity','vote_average','vote_count']:
    movies[col] = pd.to_numeric(movies[col], errors='coerce').fillna(0)

movies = movies[(movies['vote_average'] >= 0) & (movies['vote_average'] <= 10)]
movies = movies[movies['popularity'] >= 0]
movies.reset_index(drop=True, inplace=True)

print(f"Final cleaned shape  : {movies.shape}")
print(f"Remaining NaNs       : {movies.isnull().sum().sum()}")
movies.head(3)

Duplicates removed   : 1,193
Final cleaned shape  : (48806, 14)
Remaining NaNs       : 31725


,id,title,genres,overview,popularity,vote_average,vote_count,release_date,revenue,runtime,original_language,status,year,month
0,54778,Sea of Death,"Science Fiction, Thriller, Drama, TV Movie, Ac...",In the German North Sea a new cellular life fo...,1.678,4.2,13,2009-09-02,0,98,de,Released,2009.0,9.0
1,783222,Fuengirola,Unknown,Short documentary by the basque filmmaker Chum...,0.600,0.0,0,1967-01-01,0,13,es,Released,1967.0,1.0
2,51411,Ma and Pa Kettle Go to Town,Comedy,"When Pa wins a jingle-writing contest, he and ...",2.557,6.1,10,1950-04-01,0,79,en,Released,1950.0,4.0


---

## 3. Data Normalization

This section scales and standardizes key movie variables.

In [3]:
print("── Min-Max Normalization ──")
for col in ['popularity', 'vote_average', 'vote_count']:
    col_min = movies[col].min()
    col_max = movies[col].max()
    movies[f'{col}_norm'] = (movies[col] - col_min) / (col_max - col_min + 1e-9)
    print(f"  {col}_norm : min={movies[f'{col}_norm'].min():.4f}  max={movies[f'{col}_norm'].max():.4f}")

print("\n── Z-Score Standardization ──")
for col in ['popularity', 'vote_average']:
    mu = movies[col].mean()
    sigma = movies[col].std()
    movies[f'{col}_zscore'] = (movies[col] - mu) / (sigma + 1e-9)
    print(f"  {col}_zscore : mean≈{movies[f'{col}_zscore'].mean():.4f}  std≈{movies[f'{col}_zscore'].std():.4f}")

── Min-Max Normalization ──
  popularity_norm : min=0.0000  max=1.0000
  vote_average_norm : min=0.0000  max=1.0000
  vote_count_norm : min=0.0000  max=1.0000

── Z-Score Standardization ──
  popularity_zscore : mean≈0.0000  std≈1.0000
  vote_average_zscore : mean≈-0.0000  std≈1.0000


---

## 4. Exploratory Data Analysis

This section checks top movies and aggregation trends.

In [4]:
print("── Top 5 Movies by Popularity ──")
print(movies.nlargest(5, 'popularity')[['title','year','vote_average','popularity']].to_string(index=False))

print("\n── Top 5 Movies by Rating (vote_count ≥ 100) ──")
well_voted = movies[movies['vote_count'] >= 100]
print(well_voted.nlargest(5, 'vote_average')[['title','year','vote_average','vote_count']].to_string(index=False))

print("\n── GroupBy: Original Language (Top 10) ──")
if 'original_language' in movies.columns:
    lang_group = (
        movies.groupby('original_language')
              .agg(count=('title','count'),
                   avg_rating=('vote_average','mean'),
                   avg_popularity=('popularity','mean'))
              .sort_values('count', ascending=False)
              .head(10)
              .round(2)
    )
    print(lang_group)

print("\n── GroupBy: Year (2000–2024 Trends) ──")
year_group = (
    movies[(movies['year'] >= 2000) & (movies['year'] <= 2024)]
    .groupby('year')
    .agg(movies_released=('title','count'),
         avg_rating=('vote_average','mean'),
         total_popularity=('popularity','sum'))
    .sort_values('year')
    .round(2)
)
print(year_group.tail(10))

── Top 5 Movies by Popularity ──
                   title   year  vote_average  popularity
       Meg 2: The Trench 2023.0         6.912    1567.273
             Deep Throat 1972.0         5.307     157.534
      El hombre del saco 2023.0         5.083     153.607
The Shawshank Redemption 1994.0         8.702     122.610
       After We Collided 2020.0         7.202     121.032

── Top 5 Movies by Rating (vote_count ≥ 100) ──
                                title   year  vote_average  vote_count
The Three Deaths of Marisela Escobedo 2020.0         8.904         215
             The Shawshank Redemption 1994.0         8.702       24649
               Everybody’s Everything 2019.0         8.600         256
                               Psycho 1960.0         8.437        9272
                               Clouds 2020.0         8.302         953

── GroupBy: Original Language (Top 10) ──
                   count  avg_rating  avg_popularity
original_language                               

---

## 5. Exporting Dataset

This section exports the cleaned dataset to a CSV file.

In [5]:
os.makedirs('../artifacts', exist_ok=True)
movies.to_csv('../artifacts/cleaned_movies.csv', index=False)
print("✅ Cleaned dataset exported.")

reloaded = pd.read_csv('../artifacts/cleaned_movies.csv')
print(f"Reloaded shape: {reloaded.shape}")

✅ Cleaned dataset exported.
Reloaded shape: (48806, 19)
